# Process Mining & Conformance Analysis

**Data:** 10,000 government service requests with 52K+ lifecycle events  
**Activities:** Submitted > Triaged > Assigned > In Progress > Resolved > Closed (+ Escalated, Reopened)  
**Snowflake Features:** SQL analytics, `CORTEX.COMPLETE` for pattern discovery

In [ ]:
-- Set notebook context
USE DATABASE CDSB_DEMO;
USE SCHEMA RAW;
USE WAREHOUSE COMPUTE_WH;

In [ ]:
!pip install plotly --quiet

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from snowflake.snowpark.context import get_active_session

session = get_active_session()
print('Connected to', session.get_current_database(), session.get_current_schema())

## 0. Generate Service Request Event Data
Create a realistic government service request event log modelled on QLD service delivery patterns

In [ ]:
CREATE OR REPLACE TABLE SERVICE_REQUEST_EVENTS AS
WITH cases AS (
    SELECT
        SEQ4() + 1 as CASE_ID,
        DATEADD('second', UNIFORM(0, 20000000, RANDOM()), '2025-08-01'::TIMESTAMP) as BASE_TIME,
        ARRAY_CONSTRUCT('Registration Change','Payment Query','Complaint',
            'Information Request','Licence Application','Permit Application'
        )[UNIFORM(0, 5, RANDOM())]::VARCHAR as REQUEST_TYPE,
        ARRAY_CONSTRUCT('Phone','Web Portal','Email','In-Person'
        )[UNIFORM(0, 3, RANDOM())]::VARCHAR as CHANNEL,
        ARRAY_CONSTRUCT('Brisbane','Gold Coast','Sunshine Coast','Townsville',
            'Cairns','Toowoomba','Ipswich','Mackay'
        )[UNIFORM(0, 7, RANDOM())]::VARCHAR as REGION,
        ARRAY_CONSTRUCT('Handler_A','Handler_B','Handler_C','Handler_D','Handler_E'
        )[UNIFORM(0, 4, RANDOM())]::VARCHAR as HANDLER,
        UNIFORM(1, 100, RANDOM()) as FATE
    FROM TABLE(GENERATOR(ROWCOUNT => 10000))
),
events AS (
    SELECT CASE_ID, 'Submitted' as ACTIVITY, BASE_TIME as EVENT_TIME,
           REQUEST_TYPE, CHANNEL, REGION, 'System' as ACTOR FROM cases
    UNION ALL
    SELECT CASE_ID, 'Triaged',
           DATEADD('minute', UNIFORM(30, 180, RANDOM()), BASE_TIME),
           REQUEST_TYPE, CHANNEL, REGION, 'Triage_Team' FROM cases WHERE FATE > 3
    UNION ALL
    SELECT CASE_ID, 'Assigned',
           DATEADD('minute', UNIFORM(90, 360, RANDOM()), BASE_TIME),
           REQUEST_TYPE, CHANNEL, REGION, HANDLER FROM cases WHERE FATE > 5
    UNION ALL
    SELECT CASE_ID, 'In Progress',
           DATEADD('hour', UNIFORM(6, 72, RANDOM()), BASE_TIME),
           REQUEST_TYPE, CHANNEL, REGION, HANDLER FROM cases WHERE FATE > 8
    UNION ALL
    SELECT CASE_ID, 'Escalated',
           DATEADD('hour', UNIFORM(24, 120, RANDOM()), BASE_TIME),
           REQUEST_TYPE, CHANNEL, REGION, 'Escalation_Manager' FROM cases WHERE FATE BETWEEN 15 AND 30
    UNION ALL
    SELECT CASE_ID, 'Resolved',
           DATEADD('hour', UNIFORM(48, 240, RANDOM()), BASE_TIME),
           REQUEST_TYPE, CHANNEL, REGION, HANDLER FROM cases WHERE FATE > 10
    UNION ALL
    SELECT CASE_ID, 'Reopened',
           DATEADD('hour', UNIFORM(100, 400, RANDOM()), BASE_TIME),
           REQUEST_TYPE, CHANNEL, REGION, HANDLER FROM cases WHERE FATE BETWEEN 85 AND 95
    UNION ALL
    SELECT CASE_ID, 'Closed',
           DATEADD('hour', UNIFORM(72, 336, RANDOM()), BASE_TIME),
           REQUEST_TYPE, CHANNEL, REGION, HANDLER FROM cases WHERE FATE > 12
)
SELECT * FROM events ORDER BY CASE_ID, EVENT_TIME;

In [ ]:
row_count = session.sql('SELECT COUNT(*) as N FROM SERVICE_REQUEST_EVENTS').to_pandas().iloc[0,0]
case_count = session.sql('SELECT COUNT(DISTINCT CASE_ID) as N FROM SERVICE_REQUEST_EVENTS').to_pandas().iloc[0,0]
print(f'Generated {row_count:,} events across {case_count:,} cases')

## 1. Explore the Event Log
10,000 service requests with lifecycle events - modelled on QLD government service delivery

In [ ]:
SELECT ACTIVITY, COUNT(*) as EVENT_COUNT,
       COUNT(DISTINCT CASE_ID) as CASES
FROM SERVICE_REQUEST_EVENTS
GROUP BY ACTIVITY
ORDER BY EVENT_COUNT DESC

In [ ]:
df_summary = session.sql("""
    SELECT ACTIVITY, COUNT(*) as EVENT_COUNT, COUNT(DISTINCT CASE_ID) as CASES
    FROM SERVICE_REQUEST_EVENTS GROUP BY ACTIVITY ORDER BY EVENT_COUNT DESC
""").to_pandas()

fig = px.bar(df_summary, x='ACTIVITY', y='CASES', color='ACTIVITY',
             title='Cases Reaching Each Activity Stage')
fig.update_layout(template='plotly_white', showlegend=False)
fig.show()

## 2. Discover Process Paths
Use SQL to mine the actual process paths - find what really happens vs what should happen

In [ ]:
df_paths = session.sql("""
    WITH case_paths AS (
        SELECT CASE_ID,
               LISTAGG(ACTIVITY, ' > ') WITHIN GROUP (ORDER BY EVENT_TIME) as PROCESS_PATH,
               COUNT(*) as STEPS,
               DATEDIFF('hour', MIN(EVENT_TIME), MAX(EVENT_TIME)) as DURATION_HOURS,
               MAX(CHANNEL) as CHANNEL, MAX(REQUEST_TYPE) as REQUEST_TYPE
        FROM SERVICE_REQUEST_EVENTS GROUP BY CASE_ID
    )
    SELECT PROCESS_PATH, COUNT(*) as CASES,
           ROUND(AVG(DURATION_HOURS), 1) as AVG_HOURS,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 1) as PCT
    FROM case_paths GROUP BY PROCESS_PATH ORDER BY CASES DESC LIMIT 15
""").to_pandas()

print('Top 15 Process Paths Discovered:')
print(df_paths.to_string(index=False))

In [ ]:
fig = px.bar(df_paths.head(10), x='PROCESS_PATH', y='CASES',
             color='AVG_HOURS', color_continuous_scale='RdYlGn_r',
             title='Top 10 Process Paths (color = avg duration in hours)')
fig.update_layout(template='plotly_white', xaxis_tickangle=-20)
fig.show()

## 3. Conformance Analysis
Define the expected 'happy path' and detect deviations

In [ ]:
df_conformance = session.sql("""
    WITH case_paths AS (
        SELECT CASE_ID,
               LISTAGG(ACTIVITY, ' > ') WITHIN GROUP (ORDER BY EVENT_TIME) as PROCESS_PATH,
               COUNT(*) as STEPS,
               DATEDIFF('hour', MIN(EVENT_TIME), MAX(EVENT_TIME)) as DURATION_HOURS,
               MAX(CHANNEL) as CHANNEL, MAX(REQUEST_TYPE) as REQUEST_TYPE, MAX(REGION) as REGION
        FROM SERVICE_REQUEST_EVENTS GROUP BY CASE_ID
    )
    SELECT *,
        CASE
            WHEN PROCESS_PATH = 'Submitted > Triaged > Assigned > In Progress > Resolved > Closed'
                THEN 'Conformant'
            WHEN PROCESS_PATH LIKE '%Escalated%' THEN 'Escalated'
            WHEN PROCESS_PATH LIKE '%Reopened%' THEN 'Reopened'
            WHEN PROCESS_PATH NOT LIKE '%Triaged%' THEN 'Skipped Triage'
            WHEN PROCESS_PATH NOT LIKE '%Closed%' THEN 'Not Closed'
            ELSE 'Other Deviation'
        END as CONFORMANCE_STATUS
    FROM case_paths
""").to_pandas()

status_counts = df_conformance['CONFORMANCE_STATUS'].value_counts()
print('Conformance Breakdown:')
print(status_counts.to_string())

fig = px.pie(values=status_counts.values, names=status_counts.index,
             title='Process Conformance Analysis', hole=0.4)
fig.update_layout(template='plotly_white')
fig.show()

In [ ]:
fig = px.box(df_conformance, x='CONFORMANCE_STATUS', y='DURATION_HOURS',
             color='CONFORMANCE_STATUS',
             title='Resolution Time by Conformance Status (hours)')
fig.update_layout(template='plotly_white', showlegend=False)
fig.show()

avg_by_status = df_conformance.groupby('CONFORMANCE_STATUS')['DURATION_HOURS'].agg(['mean', 'median', 'count'])
print('\nAverage Duration by Conformance Status:')
print(avg_by_status.round(1).to_string())

## 4. LLM-Powered Process Discovery
Ask Cortex AI to analyse the patterns and explain what it finds

In [ ]:
path_summary = df_paths.to_string(index=False)
conformance_summary = avg_by_status.round(1).to_string()
channel_analysis = df_conformance.groupby(['CHANNEL', 'CONFORMANCE_STATUS']).size().unstack(fill_value=0)
channel_text = channel_analysis.to_string()

prompt = f"""You are a process improvement consultant for a Queensland Government department
that handles citizen service requests (licences, complaints, permits, registrations, payments).

TOP PROCESS PATHS:
{path_summary}

CONFORMANCE ANALYSIS (avg duration in hours):
{conformance_summary}

CHANNEL vs CONFORMANCE:
{channel_text}

Provide: 1) KEY FINDINGS 2) BOTTLENECKS 3) CHANNEL INSIGHTS 4) TOP 3 RECOMMENDATIONS 5) RISK FLAGS"""

llm_result = session.sql(f"""
    SELECT SNOWFLAKE.CORTEX.COMPLETE('claude-sonnet-4-6', $${prompt}$$) as ANALYSIS
""").to_pandas()
print(llm_result.iloc[0]['ANALYSIS'])

## 5. Bottleneck Analysis - Activity Transition Times

In [ ]:
df_transitions = session.sql("""
    WITH ordered_events AS (
        SELECT CASE_ID, ACTIVITY, EVENT_TIME,
               LEAD(ACTIVITY) OVER (PARTITION BY CASE_ID ORDER BY EVENT_TIME) as NEXT_ACTIVITY,
               LEAD(EVENT_TIME) OVER (PARTITION BY CASE_ID ORDER BY EVENT_TIME) as NEXT_TIME
        FROM SERVICE_REQUEST_EVENTS
    )
    SELECT ACTIVITY || ' > ' || NEXT_ACTIVITY as TRANSITION,
           COUNT(*) as FREQUENCY,
           ROUND(AVG(DATEDIFF('hour', EVENT_TIME, NEXT_TIME)), 1) as AVG_HOURS,
           ROUND(MEDIAN(DATEDIFF('hour', EVENT_TIME, NEXT_TIME)), 1) as MEDIAN_HOURS,
           ROUND(MAX(DATEDIFF('hour', EVENT_TIME, NEXT_TIME)), 1) as MAX_HOURS
    FROM ordered_events WHERE NEXT_ACTIVITY IS NOT NULL
    GROUP BY TRANSITION ORDER BY AVG_HOURS DESC
""").to_pandas()

print(df_transitions.to_string(index=False))

fig = px.bar(df_transitions, x='TRANSITION', y='AVG_HOURS',
             color='FREQUENCY', color_continuous_scale='Blues',
             title='Average Time Between Activities (hours)')
fig.update_layout(template='plotly_white', xaxis_tickangle=-30)
fig.show()

## 6. Prepare Data for Neo4j Graph Analytics
Neo4j requires **numeric-only** columns (no VARCHAR properties).
We create separate node tables per type and separate relationship tables per type.

In [ ]:
CREATE OR REPLACE TABLE PROCESS_NODES AS
SELECT ROW_NUMBER() OVER (ORDER BY NODE_TYPE, NODE_ID) as NODEID,
       NODE_ID, NODE_TYPE
FROM (
    SELECT DISTINCT CAST(CASE_ID AS VARCHAR) as NODE_ID, 'Case' as NODE_TYPE FROM SERVICE_REQUEST_EVENTS
    UNION ALL
    SELECT DISTINCT ACTOR as NODE_ID, 'Handler' as NODE_TYPE FROM SERVICE_REQUEST_EVENTS
    UNION ALL
    SELECT DISTINCT REGION as NODE_ID, 'Region' as NODE_TYPE FROM SERVICE_REQUEST_EVENTS
);

-- Neo4j node tables: NODEID only (no VARCHAR properties)
CREATE OR REPLACE TABLE CASE_NODES AS SELECT NODEID FROM PROCESS_NODES WHERE NODE_TYPE = 'Case';
CREATE OR REPLACE TABLE HANDLER_NODES AS SELECT NODEID FROM PROCESS_NODES WHERE NODE_TYPE = 'Handler';
CREATE OR REPLACE TABLE REGION_NODES AS SELECT NODEID FROM PROCESS_NODES WHERE NODE_TYPE = 'Region';

In [ ]:
-- Build full relationship table (for reference/dashboard)
CREATE OR REPLACE TABLE PROCESS_RELATIONSHIPS AS
SELECT n1.NODEID as SOURCENODEID, n2.NODEID as TARGETNODEID,
       'HANDLED_BY' as RELATIONSHIPTYPE, COUNT(*) as WEIGHT
FROM SERVICE_REQUEST_EVENTS e
JOIN PROCESS_NODES n1 ON CAST(e.CASE_ID AS VARCHAR) = n1.NODE_ID AND n1.NODE_TYPE = 'Case'
JOIN PROCESS_NODES n2 ON e.ACTOR = n2.NODE_ID AND n2.NODE_TYPE = 'Handler'
GROUP BY n1.NODEID, n2.NODEID
UNION ALL
SELECT n1.NODEID, n2.NODEID, 'IN_REGION', COUNT(*)
FROM SERVICE_REQUEST_EVENTS e
JOIN PROCESS_NODES n1 ON CAST(e.CASE_ID AS VARCHAR) = n1.NODE_ID AND n1.NODE_TYPE = 'Case'
JOIN PROCESS_NODES n2 ON e.REGION = n2.NODE_ID AND n2.NODE_TYPE = 'Region'
GROUP BY n1.NODEID, n2.NODEID;

-- Neo4j rel tables: SOURCENODEID + TARGETNODEID + FLOAT weight only
CREATE OR REPLACE TABLE HANDLED_BY_RELS AS
SELECT SOURCENODEID, TARGETNODEID, WEIGHT::FLOAT AS WEIGHT
FROM PROCESS_RELATIONSHIPS WHERE RELATIONSHIPTYPE = 'HANDLED_BY';

CREATE OR REPLACE TABLE IN_REGION_RELS AS
SELECT SOURCENODEID, TARGETNODEID, WEIGHT::FLOAT AS WEIGHT
FROM PROCESS_RELATIONSHIPS WHERE RELATIONSHIPTYPE = 'IN_REGION';

In [ ]:
for tbl in ['CASE_NODES','HANDLER_NODES','REGION_NODES','HANDLED_BY_RELS','IN_REGION_RELS']:
    n = session.sql(f'SELECT COUNT(*) as N FROM {tbl}').to_pandas().iloc[0,0]
    print(f'{tbl}: {n:,} rows')
print('\n-> Ready for Neo4j Graph Analytics (next notebook)')

## Key Takeaways

| Capability | Approach | No Specialist Tools Needed |
|-----------|----------|---------------------------|
| Process Discovery | SQL `LISTAGG` + `LEAD/LAG` | Just SQL windowing functions |
| Conformance Checking | Path comparison with `CASE` | Pattern matching in SQL |
| Bottleneck Analysis | `DATEDIFF` on transitions | Standard SQL |
| Pattern Explanation | `CORTEX.COMPLETE` | LLM interprets data inline |
| Graph Preparation | Node/relationship tables | Ready for Neo4j or native graph |

**Process mining doesn't require specialised tools - SQL + Cortex AI gets you 80% of the way**